In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as func

In [ ]:
!wget https://github.com/karpathy/char-rnn/raw/refs/heads/master/data/tinyshakespeare/input.txt

--2026-03-09 11:01:52--  https://github.com/karpathy/char-rnn/raw/refs/heads/master/data/tinyshakespeare/input.txt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt [following]
--2026-03-09 11:01:53--  https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-03-09 11:01:53 (19.4 MB/s) - ‘input.txt.2’ sav

In [ ]:
with open("input.txt") as f:
    text = f.read()

print(len(text))

1115394


In [ ]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
c2i = {ch: i for i, ch in enumerate(chars)}
i2c = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [c2i[c] for c in s]
decode = lambda l: "".join([i2c[i] for i in l])

In [ ]:
encode("PETYA\n!")

[28, 17, 32, 37, 13, 0, 2]

In [ ]:
decode(encode("PETYA\n!"))

'PETYA\n!'

In [ ]:
print(decode(encode("PETYA\n!")))

PETYA
!


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype, data[:100])

torch.Size([1115394]) torch.int64 tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
train = data[: int(0.8 * len(data))]
valid = data[int(0.8 * len(data)) :]

In [ ]:
block_size = 256
batch_size = 64


def get_batch(split):
    data_out = train if split == "train" else valid
    index = torch.randint(len(data_out) - block_size - 1, (batch_size,))
    x = torch.stack([data_out[i : i + block_size] for i in index])
    y = torch.stack([data_out[1 + i : 1 + i + block_size] for i in index])
    return x, y

In [ ]:
get_batch("train")[0].shape

torch.Size([64, 256])

In [ ]:
# Time = Sequence axis
# x: Batch, Time, Features

In [ ]:
torch.tril(torch.ones(block_size, block_size))

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 1., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size=384, n_embd=384, dropout=0.2):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, F = x.shape
        tril = torch.tril(torch.ones((block_size, block_size), device=x.device))
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        attention = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)
        attention = attention.masked_fill(tril[:T, :T] == 0, float("-inf"))

        attention = func.softmax(attention, dim=-1)
        attention = self.dropout(attention)

        return attention @ v

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads=6, head_size=384, n_embd=384, dropout=0.2):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size, n_embd, dropout) for i in range(num_heads)]
        )
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd=384, dropout=0.2):
        super().__init__()
        self.feed = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.feed(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd=384, num_heads=6):
        super().__init__()
        head_size = n_embd // num_heads
        self.multihead = MultiHeadAttention(
            num_heads=num_heads, n_embd=n_embd, head_size=head_size
        )
        self.feed = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = self.ln1(x + self.multihead(x))
        x = self.ln2(x + self.feed(x))
        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self, vocab_size, block_size=256, n_embd=384, block_count=6, num_heads=6
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd=n_embd, num_heads=num_heads) for i in range(block_count)]
        )
        self.out = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        B, T = x.shape
        token_embd = self.token_embedding(x)
        position_embd = self.position_embedding(x)
        x = token_embd + position_embd
        x = self.blocks(x)
        return self.out(x)

In [ ]:
device = "cuda"
model = Decoder(vocab_size)
model.to(device)

Decoder(
  (token_embedding): Embedding(65, 384)
  (position_embedding): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (multihead): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (feed): FeedForward(
        (feed): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      

In [ ]:
sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

10.788161

In [ ]:
from tqdm import tqdm

model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, fused=True)

max_iters = 2500
eval = 500
eval_dur = 50


def calc_loss(logits, y):
    B, T, F = logits.shape
    logits = logits.view(B * T, F)
    target = y.view(B * T)
    loss = func.cross_entropy(logits, target)
    return loss


for i in tqdm(range(max_iters)):
    if i % eval == 0 or i == max_iters - 1:
        with torch.no_grad():
            model.eval()
            losses = []
            for split in ["train", "eval"]:
                for k in range(eval_dur):
                    x, y = get_batch(split)
                    x, y = x.to(device), y.to(device)
                    logits = model(x)
                    loss = calc_loss(logits, y)
                    losses.append(loss.detach().item())
                print(f"Step {i} Split {split}: {np.mean(losses)}")
            model.train()
    x, y = get_batch("train")
    x, y = x.to(device), y.to(device)
    logits = model(x)
    loss = calc_loss(logits, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  0%|          | 0/2500 [00:00<?, ?it/s]W0309 11:02:17.097000 15682 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Step 0 Split train: 4.252672204971313
Step 0 Split eval: 4.252636523246765


 20%|██        | 500/2500 [04:45<14:04,  2.37it/s]

Step 500 Split train: 2.1328357648849487


 20%|██        | 501/2500 [05:00<2:42:42,  4.88s/it]

...

Step 2000 Split eval: 1.560510356426239


100%|█████████▉| 2499/2500 [20:04<00:00,  2.33it/s]

Step 2499 Split train: 1.3353989624977112


100%|██████████| 2500/2500 [20:20<00:00,  2.05it/s]

Step 2499 Split eval: 1.5097507870197295


In [ ]:
def generate(primer, max_tokens):
    with torch.no_grad():
        for i in tqdm(range(max_tokens)):
            primer_input = primer[:, -block_size:]
            logits = model(primer_input)
            logits = logits[:, -1, :]
            probs = func.softmax(logits, dim=-1)
            next_int = torch.multinomial(probs, num_samples=1)
            primer = torch.cat([primer, next_int], dim=-1)
    return primer


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(generate(context, max_tokens=1024)[0].tolist()))

  0%|          | 0/1024 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
100%|██████████| 1024/1024 [02:11<00:00,  7.79it/s] 


GLOUCESTER:
He okme with alliken which which the stother actions,
Who, that avocca--

GLOUCESTER:
The wilt oft, now inn the joyich of-

GLOUCESTER:
To the say oft! I musteer not the day
Coriols warw on the terpeassored
And cut mande I'll plercy: Word with hen,
Doing none more to thime of help?

HASTINGS:
We seem power I burned the fice sell to succkn'd,
Beingbramed thou dinpost her both spirpeal?
Thost cont not the Lete of thu Runtland,
Eis the legs of alll the irson's fie.
My greath, and sweet it, thou dide.
Butt, anot Were e answer so?

AUTO, as trenguk leave with vity's shout beclosies
Will thou with Lord or this trecor
I sit we lot, while atters honeste did of lost!

SICINIUS:
O, I that would the riege, I warrant is
In these goone, that by copart.

COMINIUS:
Ay, I do not tower streew you?
If pratche heart that I speak.

ANTIGONUS:
But twell shoome heaven't how the speak,
And were comes to know no the crown,
The conquire, as afare them spokets; what's mays,
Let betw-tire tide o que